# Traffic Insights - Traffic Congestion Analysis Using Apache Spark

This notebook downloads the **UCI Metro Interstate Traffic Volume** dataset (hourly traffic on I-94 westbound, Minneapolis-St Paul, 2012-2018), cleans and analyses it with **PySpark and Spark SQL**, draws charts, and exports `traffic_insights_results.zip` for the dashboard's **Import Spark Results** button.

**How to run:** `Runtime -> Run all`. Nothing needs to be edited. Takes roughly 3-6 minutes on a free Colab runtime.

> **Important - congestion levels are estimates.** The dataset only contains *traffic volume* (vehicles per hour). It has no speed, delay or travel-time data. The categories **Low / Moderate / High / Severe** below are produced from fixed volume thresholds chosen for this project (Low < 1500, Moderate 1500-3499, High 3500-4999, Severe >= 5000). They indicate how busy the road is relative to those thresholds, **not** measured congestion.

Dataset: Hogue, John (2019). *Metro Interstate Traffic Volume*. UCI Machine Learning Repository. https://doi.org/10.24432/C5X60B

## 1. Setup

In [ ]:
import os
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["HADOOP_CONF_DIR"] = r"C:\hadoop\etc\hadoop"
os.environ["PATH"] += r";C:\hadoop\bin"
os.environ["HADOOP_USER_NAME"] = os.environ.get("USERNAME", "root")
import sys; print(sys.version)

In [ ]:
# Install PySpark (+ a Java runtime that Spark 3.5 supports). Safe to re-run.
!pip -q install pyspark==3.5.3
!apt-get -qq install -y openjdk-17-jre-headless > /dev/null 2>&1 || true

In [ ]:
import os, json, gzip, shutil, zipfile, urllib.request, datetime as dt
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

WORK = "/content/traffic_work" if os.path.isdir("/content") else os.path.abspath("traffic_work")
RESULTS_DIR = os.path.join(WORK, "traffic_insights_results")
os.makedirs(RESULTS_DIR, exist_ok=True)

# Volume thresholds (vehicles per hour) used for the ESTIMATED congestion level.
# The dashboard backend and the demo data use exactly the same numbers.
LOW_MAX, MODERATE_MAX, HIGH_MAX = 1500, 3500, 5000
LEVELS = ["Low", "Moderate", "High", "Severe"]
print("Working folder:", WORK)

## 2. Download the dataset

In [ ]:
UCI_URL = "https://archive.ics.uci.edu/static/public/492/metro+interstate+traffic+volume.zip"

def download_dataset():
    path = os.path.join(WORK, "metro_interstate_traffic_volume.zip")
    req = urllib.request.Request(UCI_URL, headers={"User-Agent": "Mozilla/5.0 (TrafficInsights college project)"})
    with urllib.request.urlopen(req, timeout=90) as resp, open(path, "wb") as out:
        shutil.copyfileobj(resp, out)
    return path

def extract_csv(path):
    """Accepts the UCI .zip, a .csv.gz or a plain .csv and returns the path of a plain CSV."""
    raw_dir = os.path.join(WORK, "raw"); os.makedirs(raw_dir, exist_ok=True)
    if zipfile.is_zipfile(path):
        with zipfile.ZipFile(path) as z:
            z.extractall(raw_dir)
        candidates = [os.path.join(r, f) for r, _, fs in os.walk(raw_dir) for f in fs if f.endswith((".csv", ".csv.gz"))]
    else:
        candidates = [path]
    if not candidates:
        raise FileNotFoundError("No .csv or .csv.gz file found inside the download.")
    src = candidates[0]
    if src.endswith(".gz"):
        dst = os.path.join(raw_dir, "Metro_Interstate_Traffic_Volume.csv")
        with gzip.open(src, "rb") as fin, open(dst, "wb") as fout:
            shutil.copyfileobj(fin, fout)
        return dst
    return src

try:
    zip_path = download_dataset()
    print("Downloaded:", zip_path, f"({os.path.getsize(zip_path)/1e6:.1f} MB)")
except Exception as e:
    print("Automatic download failed:", e)
    print("Download the dataset manually from https://archive.ics.uci.edu/dataset/492 and upload the ZIP below.")
    from google.colab import files
    uploaded = files.upload()
    zip_path = os.path.join(WORK, list(uploaded.keys())[0])
    with open(zip_path, "wb") as f:
        f.write(list(uploaded.values())[0])

csv_path = extract_csv(zip_path)
print("CSV ready:", csv_path)

## 3. Start Spark and load the raw data

In [ ]:
spark = (SparkSession.builder.master("local[*]").appName("TrafficInsights")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.sql.session.timeZone", "UTC")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

schema = StructType([
    StructField("holiday", StringType()), StructField("temp", DoubleType()),
    StructField("rain_1h", DoubleType()), StructField("snow_1h", DoubleType()),
    StructField("clouds_all", IntegerType()), StructField("weather_main", StringType()),
    StructField("weather_description", StringType()), StructField("date_time", StringType()),
    StructField("traffic_volume", IntegerType()),
])
raw = spark.read.option("header", True).schema(schema).csv(csv_path)
raw.printSchema()
raw.show(5, truncate=False)
print("Raw rows:", raw.count())

## 4. Profile the raw data

Before cleaning we look for missing values and duplicate timestamps. The UCI file contains several rows for the *same hour* (one per weather description), which would count the same hour's traffic more than once if left in.

In [ ]:
null_counts = raw.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in raw.columns]).toPandas().T
null_counts.columns = ["null_values"]
print(null_counts)

total_rows = raw.count()
distinct_hours = raw.select("date_time").distinct().count()
print(f"\nRows: {total_rows:,} | distinct timestamps: {distinct_hours:,} | surplus rows sharing a timestamp: {total_rows - distinct_hours:,}")
raw.select("traffic_volume", "temp", "rain_1h", "snow_1h", "clouds_all").describe().show()

## 5. Clean and transform with PySpark

Steps: parse timestamps -> drop rows with missing timestamp/volume -> remove physically impossible values (temperature of 0 K, more than 500 mm of rain in an hour, negative volume) -> keep **one row per hour** -> convert Kelvin to Celsius -> add time features -> add the **estimated** congestion level.

In [ ]:
report = {"raw_rows": raw.count()}

df = raw.withColumn("date_time", F.to_timestamp("date_time", "yyyy-MM-dd HH:mm:ss"))
df = df.dropna(subset=["date_time", "traffic_volume"])
report["after_dropping_missing_timestamp_or_volume"] = df.count()

# Sensor/data errors: 0 Kelvin is impossible, and thousands of mm of rain in one hour is physically implausible.
impossible = F.coalesce((F.col("temp") <= 0) | (F.col("rain_1h") > 500) | (F.col("traffic_volume") < 0), F.lit(False))
df = df.filter(~impossible)
report["after_removing_impossible_values"] = df.count()

# Several rows per hour (different weather descriptions): keep exactly one, chosen deterministically.
w = Window.partitionBy("date_time").orderBy(F.col("weather_main"), F.col("weather_description"))
df = df.withColumn("_rn", F.row_number().over(w)).filter(F.col("_rn") == 1).drop("_rn")
report["after_one_row_per_hour"] = df.count()

df = df.fillna({"rain_1h": 0.0, "snow_1h": 0.0, "clouds_all": 0, "holiday": "None",
                "weather_main": "Unknown", "weather_description": "Unknown"})

vol = F.col("traffic_volume")
df = (df
      .withColumn("temp_c", F.round(F.col("temp") - 273.15, 2)).drop("temp")
      .withColumn("hour", F.hour("date_time"))
      .withColumn("day_of_week", F.date_format("date_time", "EEEE"))
      .withColumn("day_num", ((F.dayofweek("date_time") + 5) % 7) + 1)      # Monday=1 ... Sunday=7
      .withColumn("month", F.month("date_time"))
      .withColumn("year", F.year("date_time"))
      .withColumn("year_month", F.date_format("date_time", "yyyy-MM"))
      .withColumn("date", F.date_format("date_time", "yyyy-MM-dd"))
      .withColumn("congestion_level",   # ESTIMATE from volume only
                  F.when(vol < LOW_MAX, "Low").when(vol < MODERATE_MAX, "Moderate")
                   .when(vol < HIGH_MAX, "High").otherwise("Severe")))

cleaned = df.select("date_time", "holiday", "temp_c", "rain_1h", "snow_1h", "clouds_all",
                    "weather_main", "weather_description", "traffic_volume", "hour",
                    "day_of_week", "day_num", "month", "year", "year_month", "date",
                    "congestion_level").orderBy("date_time").cache()
report["final_cleaned_rows"] = cleaned.count()

print("Cleaning report")
for k, v in report.items():
    print(f"  {k:<48}{v:>10,}")
cleaned.show(5, truncate=False)
cleaned.createOrReplaceTempView("traffic")

## 6. Spark SQL analysis

In [ ]:
hourly = spark.sql("""
    SELECT hour,
           ROUND(AVG(traffic_volume), 2) AS avg_volume,
           MAX(traffic_volume)           AS max_volume,
           MIN(traffic_volume)           AS min_volume,
           COUNT(*)                      AS record_count
    FROM traffic GROUP BY hour ORDER BY hour""")

daily = spark.sql("""
    SELECT date,
           ROUND(AVG(traffic_volume), 2) AS avg_volume,
           MAX(traffic_volume)           AS max_volume,
           COUNT(*)                      AS record_count
    FROM traffic GROUP BY date ORDER BY date""")

monthly = spark.sql("""
    SELECT year_month,
           ROUND(AVG(traffic_volume), 2) AS avg_volume,
           MAX(traffic_volume)           AS max_volume,
           COUNT(*)                      AS record_count
    FROM traffic GROUP BY year_month ORDER BY year_month""")

weekday = spark.sql("""
    SELECT day_num, day_of_week,
           ROUND(AVG(traffic_volume), 2) AS avg_volume,
           COUNT(*)                      AS record_count
    FROM traffic GROUP BY day_num, day_of_week ORDER BY day_num""")

congestion = spark.sql("""
    SELECT congestion_level,
           COUNT(*) AS record_count,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS percentage
    FROM traffic GROUP BY congestion_level
    ORDER BY CASE congestion_level WHEN 'Low' THEN 1 WHEN 'Moderate' THEN 2 WHEN 'High' THEN 3 ELSE 4 END""")

weather = spark.sql("""
    SELECT weather_main,
           ROUND(AVG(traffic_volume), 2) AS avg_volume,
           COUNT(*)                      AS record_count
    FROM traffic GROUP BY weather_main ORDER BY avg_volume DESC""")

print("Hourly pattern"); hourly.show(24)
print("Day of week"); weekday.show()
print("Estimated congestion distribution (volume-based estimate)"); congestion.show()
print("Average volume by weather"); weather.show()
print("Top 10 busiest hours on record")
spark.sql("SELECT date_time, traffic_volume, weather_main, congestion_level FROM traffic ORDER BY traffic_volume DESC LIMIT 10").show(truncate=False)

## 7. Summary statistics

In [ ]:
stats = spark.sql("""
    SELECT COUNT(*) AS total_records,
           ROUND(AVG(traffic_volume), 2) AS avg_traffic_volume,
           MAX(traffic_volume) AS max_traffic_volume,
           MIN(traffic_volume) AS min_traffic_volume,
           MIN(date) AS date_start, MAX(date) AS date_end
    FROM traffic""").first().asDict()

hourly_pd = hourly.toPandas()
weekday_pd = weekday.toPandas()
monthly_pd = monthly.toPandas()
daily_pd = daily.toPandas()
congestion_pd = congestion.toPandas()
weather_pd = weather.toPandas()

median, q1, q3 = cleaned.approxQuantile("traffic_volume", [0.5, 0.25, 0.75], 0.001)
corr_temp = cleaned.stat.corr("temp_c", "traffic_volume")

summary = {
    "total_records": int(stats["total_records"]),
    "avg_traffic_volume": float(stats["avg_traffic_volume"]),
    "max_traffic_volume": int(stats["max_traffic_volume"]),
    "min_traffic_volume": int(stats["min_traffic_volume"]),
    "median_traffic_volume": float(median),
    "peak_hour": int(hourly_pd.loc[hourly_pd["avg_volume"].idxmax(), "hour"]),
    "quietest_hour": int(hourly_pd.loc[hourly_pd["avg_volume"].idxmin(), "hour"]),
    "busiest_day_of_week": str(weekday_pd.loc[weekday_pd["avg_volume"].idxmax(), "day_of_week"]),
    "busiest_month": str(monthly_pd.loc[monthly_pd["avg_volume"].idxmax(), "year_month"]),
    "date_start": str(stats["date_start"]),
    "date_end": str(stats["date_end"]),
    "correlation_temp_vs_volume": round(float(corr_temp), 4),
    "congestion_thresholds": {"Low": "< 1500", "Moderate": "1500-3499", "High": "3500-4999", "Severe": ">= 5000"},
    "cleaning_report": report,
    "source": "UCI Metro Interstate Traffic Volume (dataset 492), processed with PySpark " + spark.version,
    "generated_at": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
    "notes": "Congestion levels are ESTIMATES derived only from traffic volume, not measured speeds or delays.",
}
print(json.dumps(summary, indent=2))

## 8. Charts

Remember: the congestion chart shows an **estimate based on volume thresholds**.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
fig.suptitle("Traffic Insights - Metro Interstate Traffic Volume (PySpark results)", fontsize=15, fontweight="bold")

axes[0, 0].plot(hourly_pd["hour"], hourly_pd["avg_volume"], marker="o", color="#2563eb")
axes[0, 0].set(title="Average traffic by hour of day", xlabel="Hour", ylabel="Vehicles per hour", xticks=range(0, 24, 2))

axes[0, 1].bar(weekday_pd["day_of_week"].str[:3], weekday_pd["avg_volume"], color="#2563eb")
axes[0, 1].set(title="Average traffic by day of week", ylabel="Vehicles per hour")

axes[0, 2].plot(pd.to_datetime(monthly_pd["year_month"]), monthly_pd["avg_volume"], color="#2563eb")
axes[0, 2].set(title="Monthly average traffic", ylabel="Vehicles per hour")
axes[0, 2].tick_params(axis="x", rotation=30)

colors = ["#2e9e6b", "#d9a400", "#e8782a", "#c8383d"]
axes[1, 0].pie(congestion_pd["record_count"], labels=congestion_pd["congestion_level"], colors=colors, autopct="%1.1f%%", startangle=90)
axes[1, 0].set_title("Estimated congestion levels (volume-based)")

axes[1, 1].barh(weather_pd["weather_main"], weather_pd["avg_volume"], color="#2563eb")
axes[1, 1].set(title="Average traffic by weather", xlabel="Vehicles per hour")
axes[1, 1].invert_yaxis()

daily_pd["date"] = pd.to_datetime(daily_pd["date"])
axes[1, 2].plot(daily_pd["date"], daily_pd["avg_volume"], linewidth=0.6, color="#2563eb")
axes[1, 2].set(title="Daily average traffic (gaps = missing days in the dataset)", ylabel="Vehicles per hour")

plt.tight_layout()
plt.show()

## 9. Export `traffic_insights_results.zip`

The ZIP contains exactly the six files the dashboard expects. The cell re-opens the ZIP and checks it before offering the download.

In [ ]:
REQUIRED = {
    "hourly_trends.csv": ["hour", "avg_volume", "max_volume", "min_volume", "record_count"],
    "daily_trends.csv": ["date", "avg_volume", "max_volume", "record_count"],
    "monthly_trends.csv": ["year_month", "avg_volume", "max_volume", "record_count"],
    "congestion_distribution.csv": ["congestion_level", "record_count", "percentage"],
    "cleaned_traffic.csv": ["date_time", "traffic_volume", "congestion_level", "holiday", "temp_c", "weather_main"],
}

daily_pd["date"] = daily_pd["date"].dt.strftime("%Y-%m-%d")
hourly_pd.to_csv(os.path.join(RESULTS_DIR, "hourly_trends.csv"), index=False)
daily_pd.to_csv(os.path.join(RESULTS_DIR, "daily_trends.csv"), index=False)
monthly_pd.to_csv(os.path.join(RESULTS_DIR, "monthly_trends.csv"), index=False)
congestion_pd.to_csv(os.path.join(RESULTS_DIR, "congestion_distribution.csv"), index=False)
# Write the timestamp as plain text so pandas never has to do time-zone conversion
(cleaned.withColumn("date_time", F.date_format("date_time", "yyyy-MM-dd HH:mm:ss"))
        .toPandas().to_csv(os.path.join(RESULTS_DIR, "cleaned_traffic.csv"), index=False))
with open(os.path.join(RESULTS_DIR, "traffic_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

ZIP_PATH = os.path.join(WORK, "traffic_insights_results.zip")
FILES = ["traffic_summary.json", "hourly_trends.csv", "daily_trends.csv", "monthly_trends.csv",
         "congestion_distribution.csv", "cleaned_traffic.csv"]
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for name in FILES:
        z.write(os.path.join(RESULTS_DIR, name), arcname=name)   # files at the top level of the ZIP

# --- self-check
with zipfile.ZipFile(ZIP_PATH) as z:
    names = set(z.namelist())
    missing = [n for n in FILES if n not in names]
    assert not missing, f"Missing files in ZIP: {missing}"
    for name, cols in REQUIRED.items():
        head = pd.read_csv(z.open(name), nrows=5)
        absent = [c for c in cols if c not in head.columns]
        assert not absent, f"{name} is missing columns {absent}"
        print(f"OK  {name:<30} columns: {len(head.columns)}")
    json.loads(z.read("traffic_summary.json"))
print(f"\nCreated {ZIP_PATH} ({os.path.getsize(ZIP_PATH)/1e6:.1f} MB) - ready to import into the dashboard.")

In [ ]:
try:
    from google.colab import files
    files.download(ZIP_PATH)          # downloads traffic_insights_results.zip to your computer
except ImportError:
    print("Not running in Colab - the ZIP is at:", ZIP_PATH)

## 10. Notes and limitations

* **Congestion levels are estimates.** They are derived only from hourly traffic *volume* with fixed thresholds. They are not measured speeds, delays or travel times, and a different threshold choice would change the distribution.
* The dataset covers **one road segment** (I-94 westbound near Minneapolis-St Paul) and has **gaps** (missing hours/days), so daily and monthly averages are based on different numbers of records.
* Where several rows shared one timestamp, one row was kept (alphabetically first weather label), so the weather label for such hours is approximate. Traffic volume is the same across those rows.
* Rows with impossible values (0 K temperature, > 500 mm rain per hour) were removed; the counts are in the cleaning report above and in `traffic_summary.json`.

**Next step:** open the dashboard (`http://127.0.0.1:8000`), go to **Import & Data**, and upload `traffic_insights_results.zip`.